# TermNorm Backend

Connect to TermNorm, sync experiments, replay pipelines, compare variants.

**Prerequisites:** TermNorm running at `http://127.0.0.1:8000`

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path("..").resolve()
sys.path.insert(0, str(PROJECT_ROOT))

from api.models.backend import BackendConnection
from api.services.project_store import ProjectStore
from api.services.backend_client import BackendClient

TERMNORM_URL = "http://127.0.0.1:8000"
BACKEND_ID = "termnorm-local"

store = ProjectStore(base_dir=PROJECT_ROOT / ".promptpotter" / "projects")
client = BackendClient(TERMNORM_URL)

print(f"Project root: {PROJECT_ROOT}")
print(f"Store: {store.base_dir}")
print("Ready")

## 1. Register backend

In [ ]:
# Register (idempotent — skips if already exists)
if store.get_backend(BACKEND_ID):
    backend = store.get_backend(BACKEND_ID)
    print(f"Already registered: {backend.name} ({backend.base_url})")
else:
    backend = BackendConnection(
        id=BACKEND_ID,
        name="TermNorm Local",
        backend_type="termnorm",
        base_url=TERMNORM_URL,
    )
    store.register_backend(backend)
    print(f"Registered: {backend.name}")

print(f"Store: .promptpotter/projects/{BACKEND_ID}/")

## 2. Sync experiments from TermNorm

In [ ]:
count = await client.sync_experiments(store, BACKEND_ID)

# Update last_synced_at
from datetime import datetime, timezone
backend.last_synced_at = datetime.now(timezone.utc).isoformat()
store.update_backend(backend)

print(f"Synced {count} experiment(s)")

In [ ]:
# Peek at what we got (native TermNorm format)
experiments = store.load_sync(BACKEND_ID, "experiments.json")
for exp in experiments.get("experiments", []):
    exp_id = exp.get("experiment_id", exp.get("id", "?"))
    print(f"  {exp_id}: {exp.get('name', '')} — {exp.get('description', '')[:80]}")

## 3. Inspect a synced experiment

In [ ]:
EXPERIMENT_ID = "1_production_historical"  # adjust if needed

exp_data = store.load_sync(BACKEND_ID, f"experiments/{EXPERIMENT_ID}.json")

terms = client.extract_session_terms(exp_data)
queries = client.extract_replay_queries(exp_data)

print(f"Experiment: {exp_data.get('experiment', {}).get('name', EXPERIMENT_ID)}")
print(f"Mappings: {len(exp_data.get('mappings', []))}")
print(f"Session terms: {len(terms)}")
print(f"Replayable queries (with ground truth): {len(queries)}")
print()
print("First 5 queries:")
for q in queries[:5]:
    print(f"  {q['query'][:60]}  ->  GT: {q['ground_truth'][:50]}")

In [ ]:
# Experiment overview
exp_meta = exp_data.get("experiment", {})
runs = exp_data.get("runs", [])

print("EXPERIMENT OVERVIEW")
print("=" * 60)
print(f"  ID:              {exp_meta.get('experiment_id', '?')}")
print(f"  Name:            {exp_meta.get('name', '?')}")
print(f"  Description:     {exp_meta.get('description', '?')}")
print(f"  Lifecycle stage: {exp_meta.get('lifecycle_stage', '?')}")
print(f"  Mappings count:  {exp_data.get('mappings_count', len(exp_data.get('mappings', [])))}")
print(f"  Runs:            {len(runs)}")

if exp_meta.get("creation_time"):
    from datetime import datetime
    created = datetime.fromtimestamp(exp_meta["creation_time"] / 1000)
    print(f"  Created:         {created.isoformat()}")

for i, r in enumerate(runs):
    print(f"\n  Run {i}: {r.get('run_name', r.get('run_id', '?'))}")
    print(f"    ID:     {r.get('run_id', '?')}")
    print(f"    Status: {r.get('status', '?')}")
    print(f"    Tags:   {r.get('tags', {})}")

In [ ]:
import pandas as pd

run = exp_data["runs"][0]

# Run parameters
print("RUN PARAMETERS")
print("-" * 40)
params_df = pd.DataFrame(
    [{"parameter": k, "value": v} for k, v in run["params"].items()]
)
display(params_df)

# Pipeline config
pipeline = run.get("pipeline", {})
config = pipeline.get("config", {})
steps = config.get("steps", [])

print(f"\nPIPELINE: {config.get('name', '?')} ({config.get('version', '?')})")
print(f"Description: {config.get('description', '')}")
print(f"Notation: {pipeline.get('notation', '?')}")
print()

# Pipeline steps table
step_rows = []
for s in steps:
    step_rows.append({
        "step": s["name"],
        "type": s["type"],
        "inputs": ", ".join(s.get("signature", {}).get("input_fields", [])),
        "outputs": ", ".join(s.get("signature", {}).get("output_fields", [])),
        "model": s.get("config", {}).get("model", "—"),
        "temperature": s.get("config", {}).get("temperature", "—"),
    })
steps_df = pd.DataFrame(step_rows)
display(steps_df)

In [ ]:
# Run metrics (what TermNorm already measured — full pipeline with LLM2)
metrics = run["metrics"]

print("RUN METRICS (baseline — full pipeline with LLM2)")
print("=" * 50)
print(f"  Queries evaluated:  {int(metrics.get('num_queries', 0))}")
print(f"  hit@1 (Accuracy):   {metrics.get('hit_at_1', 0):.1%}")
print(f"  hit@3:              {metrics.get('hit_at_3', 0):.1%}")
print(f"  hit@5:              {metrics.get('hit_at_5', 0):.1%}")
print(f"  MRR:                {metrics.get('mrr', 0):.3f}")
print(f"  Avg latency:        {metrics.get('avg_latency_ms', 0):,.0f} ms")
print(f"  Avg confidence:     {metrics.get('avg_confidence', 0):.3f}")

In [ ]:
# Evaluation results from the stored run
eval_results = run.get("evaluation_results", [])
eval_df = pd.DataFrame(eval_results)

print(f"EVALUATION RESULTS: {len(eval_df)} queries")
print("=" * 50)

if not eval_df.empty:
    print(f"\nLatency (ms):")
    print(f"  Mean:     {eval_df['latency_ms'].mean():,.0f}")
    print(f"  Median:   {eval_df['latency_ms'].median():,.0f}")
    print(f"  Min:      {eval_df['latency_ms'].min():,.0f}")
    print(f"  Max:      {eval_df['latency_ms'].max():,.0f}")

    print(f"\nConfidence:")
    print(f"  Mean:     {eval_df['confidence'].mean():.3f}")
    print(f"  Non-zero: {(eval_df['confidence'] > 0).sum()} / {len(eval_df)}")

    print(f"\nMethods: {eval_df['method'].value_counts().to_dict()}")

    print(f"\nFirst 10 results:")
    display(eval_df[["query", "predicted", "method", "confidence", "latency_ms"]].head(10))

In [ ]:
# Mappings overview
mappings = exp_data.get("mappings", [])
total = len(mappings)
with_gt = sum(1 for m in mappings if m.get("dataset_entry", "").strip() not in ("", "--"))
no_gt = total - with_gt

print(f"MAPPINGS OVERVIEW")
print("=" * 50)
print(f"  Total mappings:          {total}")
print(f"  With ground truth:       {with_gt}")
print(f"  Without ground truth:    {no_gt}  (empty or '--')")
if total:
    print(f"  Evaluation coverage:     {with_gt/total:.1%}")

# Sample mappings with ground truth
print(f"\nSample mappings (first 10 with ground truth):")
sample_rows = []
for m in mappings:
    if m.get("dataset_entry", "").strip() not in ("", "--"):
        sample_rows.append({
            "bom_material": m["bom_material"][:50],
            "dataset_entry": m["dataset_entry"][:60],
        })
    if len(sample_rows) >= 10:
        break
display(pd.DataFrame(sample_rows))

## 4. Replay (execute via TermNorm API)

Calls TermNorm's `/sessions` and `/matches` endpoints with `skip_llm_ranking=True`.

In [ ]:
import uuid
from tqdm.auto import tqdm
from api.models.backend import Execution, ExecutionResultItem

LIMIT = 0  # set to 0 for all queries, or a smaller number for quick tests
replay_queries_list = queries[:LIMIT] if LIMIT else queries
total = len(replay_queries_list)

execution_id = uuid.uuid4().hex[:12]

print(f"Replaying {total} queries against {TERMNORM_URL}...")
print(f"Execution ID: {execution_id}")
print()

# Progress tracking state
_hits = 0
_pbar = tqdm(total=total, desc="Replay", unit="query")

async def on_result(result, index, total):
    """Save result incrementally and display rich feedback."""
    global _hits
    store.append_result(BACKEND_ID, execution_id, result)

    pred = result.get("predicted", "?")
    gt = result["ground_truth"]
    q = result["query"]
    latency = result.get("latency_ms", 0)
    conf = result.get("confidence", 0)
    hit = pred == gt

    if hit:
        _hits += 1

    tag = "HIT " if hit else "MISS"
    done = index + 1
    acc = _hits / done * 100

    tqdm.write(
        f"[{done}/{total}] {tag}  {q[:50]:<50s} "
        f"| pred: {pred[:35]:<35s} | GT: {gt[:35]:<35s} "
        f"| {latency:,.0f}ms | conf: {conf:.2f} "
        f"| Running: {_hits}/{done} ({acc:.1f}%)"
    )
    _pbar.update(1)

results = await client.replay_queries(
    queries=replay_queries_list,
    terms=terms,
    skip_llm_ranking=True,
    delay_between=2.0,
    on_result=on_result,
)
_pbar.close()

# Finalize: merge .jsonl into proper Execution .json
successful = sum(1 for r in results if r["status"] == "success")
errors = sum(1 for r in results if r["status"] == "error")
total_latency = sum(r.get("latency_ms", 0) for r in results)
total_conf = sum(r.get("confidence", 0) for r in results)

execution = Execution(
    execution_id=execution_id,
    backend_id=BACKEND_ID,
    experiment_id=EXPERIMENT_ID,
    variant_label="LLM1-TokenMatching (no LLM2)",
    pipeline_notation="LLM1-TokenMatching",
    session_terms_count=len(terms),
    query_count=len(results),
    successful_count=successful,
    error_count=errors,
    results=[ExecutionResultItem(**r) for r in results],
)
store.finalize_execution(execution)

# End summary
print()
print("=" * 60)
print(f"  Accuracy (hit@1): {_hits}/{total} ({_hits/total*100:.1f}%)")
print(f"  Avg latency:      {total_latency/total:,.0f} ms")
print(f"  Avg confidence:   {total_conf/total:.3f}")
print(f"  Errors:           {errors}")
print(f"  Execution ID:     {execution_id}")
print("=" * 60)

In [ ]:
# Quick results table
import pandas as pd

rows = []
for r in results:
    gt = r["ground_truth"]
    pred = r.get("predicted", "")
    rows.append({
        "query": r["query"][:50],
        "predicted": pred[:50],
        "ground_truth": gt[:50],
        "correct": pred == gt,
        "latency_ms": r.get("latency_ms", 0),
    })

df = pd.DataFrame(rows)
display(df)

## 5. Compare variants

In [ ]:
from api.services.comparison import compute_comparison
import json

comparison = compute_comparison(
    results,
    metadata={
        "pipeline_notation": execution.pipeline_notation,
        "variant_label": execution.variant_label,
        "session_terms_count": execution.session_terms_count,
    },
)

m = comparison["metrics"]
c = comparison["classification"]
n = comparison["dataset"]["query_count"]

print(f"Queries: {n}")
print(f"")
print(f"Accuracy (hit@1):")
print(f"  Variant A (no LLM2): {m['hit_at_1']['a_count']}/{n} ({m['hit_at_1']['a']:.1%})")
print(f"  Variant B (full):    {m['hit_at_1']['b_count']}/{n} ({m['hit_at_1']['b']:.1%})")
print(f"")
print(f"Classification:")
print(f"  Both correct:   {c['both_correct']}")
print(f"  A-only correct: {c['a_only_correct']}  (LLM2 hurt)")
print(f"  B-only correct: {c['b_only_correct']}  (LLM2 helped)")
print(f"  Both wrong:     {c['both_wrong']}")

## 6. Browse stored executions

In [ ]:
executions = store.list_executions(BACKEND_ID)
for ex in executions:
    print(f"  {ex['execution_id']}  {ex['variant_label']}  "
          f"queries={ex['query_count']}  success={ex['successful_count']}  "
          f"{ex['created_at']}")